<a href="https://www.kaggle.com/code/emrebaranarca/who-live-titanic?scriptVersionId=298401539" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
train.head()

In [ ]:
train.info()
train.describe()

print(f"📊 Veri seti boyutu: {train.shape[0]} satır, {train.shape[1]} sütun")
print(f"\n🏷️  Sütunlar: {train.columns.tolist()}")

In [ ]:
train.isnull().sum()

In [ ]:
print("\n" + "="*60)
print("1️⃣  KEŞİFSEL VERİ ANALİZİ (EDA)")
print("="*60)

# --- İlk bakış ---
print("\n📋 İlk 5 satır:")
print(train.head())

print("\n📋 Veri tipleri ve eksik değerler:")
print(train.info())

print("\n📋 Temel istatistikler:")
print(train.describe())


In [ ]:
# --- Eksik Değer Analizi ---
print("\n❓ Eksik Değer Sayıları:")
missing=train.isnull().sum()
missing_pct = (train.isnull().sum() / len(train) * 100).round(1)
missing_df = pd.DataFrame({'Eksik Sayı': missing, 'Yüzde (%)': missing_pct})
print(missing_df[missing_df['Eksik Sayı'] > 0].sort_values('Yüzde (%)', ascending=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('🔍 Titanic Veri Seti - Keşifsel Analiz', fontsize=16, fontweight='bold')


survived_counts = train['Survived'].value_counts()
axes[0, 0].bar(['Hayatını Kaybetti (0)', 'Hayatta Kaldı (1)'],
               survived_counts.values, color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0, 0].set_title('Hayatta Kalma Dağılımı')
axes[0, 0].set_ylabel('Kişi Sayısı')
for i, v in enumerate(survived_counts.values):
    axes[0, 0].text(i, v + 10, str(v), ha='center', fontweight='bold')


# 2. Cinsiyet ve hayatta kalma
sex_survival = train.groupby(['Sex', 'Survived']).size().unstack(fill_value=0)
sex_survival.plot(kind='bar', ax=axes[0, 1], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0, 1].set_title('Cinsiyet ve Hayatta Kalma')
axes[0, 1].set_xticklabels(['Kadın', 'Erkek'], rotation=0)
axes[0, 1].legend(['Hayatını Kaybetti', 'Hayatta Kaldı'])

pclass_survival = train.groupby(['Pclass', 'Survived']).size().unstack(fill_value=0)
pclass_survival.plot(kind='bar', ax=axes[0, 2], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0, 2].set_title('Bilet Sınıfı ve Hayatta Kalma')
axes[0, 2].set_xticklabels(['1. Sınıf', '2. Sınıf', '3. Sınıf'], rotation=0)
axes[0, 2].legend(['Hayatını Kaybetti', 'Hayatta Kaldı'])


train['Age'].dropna().hist(bins=30, ax=axes[1, 0], color='#3498db', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Yaş Dağılımı')
axes[1, 0].set_xlabel('Yaş')
axes[1, 0].set_ylabel('Kişi Sayısı')
axes[1, 0].axvline(train['Age'].median(), color='red', linestyle='--', label=f"Medyan: {train['Age'].median():.0f}")
axes[1, 0].legend()


train['Fare'].dropna().hist(bins=30, ax=axes[1, 1], color='#9b59b6', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Bilet Ücreti Dağılımı')
axes[1, 1].set_xlabel('Ücret')
axes[1, 1].set_ylabel('Kişi Sayısı')

embarked_survival = train.groupby(['Embarked', 'Survived']).size().unstack(fill_value=0)
embarked_survival.plot(kind='bar', ax=axes[1, 2], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[1, 2].set_title('Biniş Limanı ve Hayatta Kalma')
axes[1, 2].legend(['Hayatını Kaybetti', 'Hayatta Kaldı'])
axes[1, 2].set_xticklabels(axes[1, 2].get_xticklabels(), rotation=0)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
numeric_cols = train.select_dtypes(include=[np.number]).columns
corr_matrix = train[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            square=True, ax=ax, linewidths=0.5)
ax.set_title('📊 Korelasyon Matrisi', fontsize=14, fontweight='bold')
plt.tight_layout()

In [ ]:
# --- EDA'dan Öğrendiklerimiz ---
print("""
╔══════════════════════════════════════════════════════════════╗
║  📝 EDA'DAN ÖNEMLİ BULGULAR:                                ║
║                                                              ║
║  • Kadınların hayatta kalma oranı erkeklerden çok daha yüksek║
║  • 1. sınıf yolcuların hayatta kalma şansı daha fazla       ║
║  • Yaş ve ücret önemli faktörler                             ║
║  • 'cabin' sütununda çok fazla eksik veri var (%77)          ║
║  • Bu bulgular feature engineering için yol gösterici!        ║
╚══════════════════════════════════════════════════════════════╝
""")

In [ ]:
print("="*60)
print("2️⃣  VERİ ÖN İŞLEME (PREPROCESSING)")
print("="*60)

In [ ]:
print("\n" + "="*60)
print("3️⃣  FEATURE ENGINEERING")
print("="*60)

In [ ]:
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
print("\n📌 Unvan dağılımı:")
print(train['Title'].value_counts())

      # Nadir unvanları gruplama
title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Mme': 'Mrs', 'Ms': 'Miss', 'Lady': 'Rare',
    'Sir': 'Rare', 'Capt': 'Rare', 'Don': 'Rare', 'Dona': 'Rare',
    'Jonkheer': 'Rare', 'Countess': 'Rare'
}
train['Title'] = train['Title'].map(title_mapping).fillna('Rare')
print("\n📌 Gruplanmış unvan dağılımı:")
print(train['Title'].value_counts())

In [ ]:
# --- 2. Aile Büyüklüğü ---
train['Family_size'] = train['SibSp'] + train['Parch'] + 1
train['Is_alone'] = (train['Family_size'] == 1).astype(int)
print(f"\n👨‍👩‍👧‍👦 Aile büyüklüğü aralığı: {train['Family_size'].min()} - {train['Family_size'].max()}")
print(f"👤 Yalnız yolcu oranı: {train['Is_alone'].mean()*100:.1f}%")


In [ ]:
# --- 3. Cabin bilgisi var mı? ---
train['Has_cabin'] = train['Cabin'].notna().astype(int)
print(f"\n🏠 Cabin bilgisi olan yolcu oranı: {train['Has_cabin'].mean()*100:.1f}%")

In [ ]:
# --- 4. Yaş Grubu ---
# Önce eksik yaşları dolduralım (unvan medyanı ile - daha akıllı bir yöntem)
for title in train['Title'].unique():
    median_age = train[train['Title'] == title]['Age'].median()
    train.loc[(train['Age'].isnull()) & (train['Title'] == title), 'Age'] = median_age

train['Age_group'] = pd.cut(train['Age'], bins=[0, 12, 18, 35, 60, 100],
                            labels=['Cocuk', 'Genc', 'Yetiskin', 'OrtaYas', 'Yasli'])
print(f"\n🎂 Yaş grupları dağılımı:")
print(train['Age_group'].value_counts())

In [ ]:
# --- 5. Ücret kategorisi ---
train['Fare'] = train['Fare'].fillna(train['Fare'].median())
train['Fare_category'] = pd.qcut(train['Fare'], q=4, labels=['Düşük', 'Orta', 'Yüksek', 'Çok Yüksek'])

In [ ]:
# --- 6. Eksik embarked değerini doldur ---
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

In [ ]:
# İsim ve cabin artık gerekli değil
train = train.drop(columns=['Name', 'Cabin'])

In [ ]:
print("\n📋 Son veri seti sütunları:")
print(train.columns.tolist())
print(f"\n📋 Kalan eksik değerler: {train.isnull().sum().sum()}")

In [ ]:
import matplotlib.pyplot as plt

# --- Feature Engineering Görselleştirme ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(' Feature Engineering Sonuçları', fontsize=14, fontweight='bold')

# Unvan ve hayatta kalma
title_surv = train.groupby('Title')['Survived'].mean().sort_values(ascending=False)
title_surv.plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='black')
axes[0].set_title('Unvana Göre Hayatta Kalma Oranı')
axes[0].set_ylabel('Hayatta Kalma Oranı')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

# Aile büyüklüğü ve hayatta kalma
family_surv = train.groupby('Family_size')['Survived'].mean()
family_surv.plot(kind='bar', ax=axes[1], color='#e67e22', edgecolor='black')
axes[1].set_title('Aile Büyüklüğüne Göre Hayatta Kalma')
axes[1].set_ylabel('Hayatta Kalma Oranı')


# Yaş grubu ve hayatta kalma
age_surv = train.groupby('Age_group')['Survived'].mean()
age_surv.plot(kind='bar', ax=axes[2], color='#2ecc71', edgecolor='black')
axes[2].set_title('Yaş Grubuna Göre Hayatta Kalma')
axes[2].set_ylabel('Hayatta Kalma Oranı')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45)

In [ ]:
print("\n" + "="*60)
print(" MODEL İÇİN VERİ HAZIRLAMA")
print("="*60)

In [ ]:
# Kategorik değişkenleri encode et
data_encoded = pd.get_dummies(train, columns=['Sex', 'Embarked', 'Title', 'Age_group', 'Fare_category'],
drop_first=True)
data_encoded = data_encoded.drop('Ticket',axis=1, errors='ignore')
data_encoded.head()

In [ ]:
X = data_encoded.drop('Survived', axis=1)
y = data_encoded['Survived']
X.head()


In [ ]:
print(f"\n Feature sayısı: {X.shape[1]}") # Kaç tane ipucu (sütun) var?
print(f" Örnek sayısı: {X.shape[0]}") # Kaç yolcu (satır) var?
# ... (Dağılım oranları)
print(f"\n Hedef değişken dağılımı:")
print(f" Hayatını Kaybetti (0): {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f" Hayatta Kaldı (1): {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, train_size=0.8, random_state=42, shuffle=True, stratify=y)

print(f"\n Eğitim seti: {X_train.shape[0]} örnek")
print(f" Test seti: {X_test.shape[0]} örnek")

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
print(X_train.select_dtypes(include=['object']).columns)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(" Veri hazırlama tamamlandı!")

In [ ]:
print("\n" + "="*60)
print(" MODEL EĞİTİMİ VE KARŞILAŞTIRMA")
print("="*60)
results = {}

In [ ]:
#first step deep-learning
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 1. VERİ HAZIRLIĞI (TENSOR DÖNÜŞÜMÜ)


X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1) # Boyutu (N, 1) yapıyoruz
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1, 1)


class TitanicModel(nn.Module):

    def __init__(self,input_features):

        super(TitanicModel, self).__init__()
        
        # Katmanları Tanımlıyoruz (İnşaat Malzemeleri)
        # Giriş Katmanı -> Gizli Katman 1 (64 Nöron)
        self.layer1 = nn.Linear(input_features, 64)
        
        # Gizli Katman 1 -> Gizli Katman 2 (32 Nöron)
        self.layer2 = nn.Linear(64, 32)
        
        # Gizli Katman 2 -> Çıkış Katmanı (1 Nöron - Çünkü sonuç 0 veya 1)
        self.output = nn.Linear(32, 1)
        
        # Aktivasyon Fonksiyonları (Karar Mekanizmaları)
        self.relu = nn.ReLU()      # Negatifleri sıfırlar (Düşünmeyi sağlar)
        self.sigmoid = nn.Sigmoid() # Sonucu 0 ile 1 arasına sıkıştırır (Olasılık)
        
        # Dropout (Ezberlemeyi önlemek için nöronların %20'sini rastgele kapatır)
        self.dropout = nn.Dropout(p=0.2)


    def forward(self, x):
        # Verinin model içindeki yolculuğu (İleri Yayılım)
        x = self.relu(self.layer1(x)) # Katman 1 + ReLU
        x = self.dropout(x)           # Rastgele nöron kapatma
        x = self.relu(self.layer2(x)) # Katman 2 + ReLU
        x = self.sigmoid(self.output(x)) # Çıkış + Sigmoid
        return x

input_dim=X_train_tensor.shape[1]
model=TitanicModel(input_dim)

print("✅ Model Mimarisi Kuruldu:\n", model)

#train loop

criterion = nn.BCELoss()

# Öğretmen (Optimizer): Adam (Hatalardan ders çıkarıp ağırlıkları güncelleyen algoritma)
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 200  # Modeli 200 kere eğiteceğiz
print(f"\n🚀 Eğitim Başlıyor ({epochs} Epoch)...")

for epoch in range(epochs):
    # a) Tahmin Yap (Forward Pass)
    y_pred = model(X_train_tensor)
    
    # b) Hatayı Ölç (Loss Calculation)
    loss = criterion(y_pred, y_train_tensor)
    
    # c) Geçmişi Temizle, Türev Al, Güncelle (Backward Pass)
    optimizer.zero_grad() # Eski gradyanları sıfırla
    loss.backward()       # Hatanın kaynağını bul (Türev)
    optimizer.step()      # Ağırlıkları güncelle (Öğrenme gerçekleşti)
    
    # Her 20 turda bir durumu raporla
    if (epoch+1) % 20 == 0:
        print(f'Tur {epoch+1:03}: Hata Oranı (Loss) = {loss.item():.4f}')

print("\n📊 Değerlendirme Yapılıyor...")
# Modeli değerlendirme moduna al (Dropout'u kapatır)
model.eval()

with torch.no_grad(): # Hafızayı yormamak için gradyan hesaplamayı kapat
    # Test seti üzerinde tahmin yap
    y_pred_proba_tensor = model(X_test_tensor)
    
    # Tensor'u tekrar numpy array'e çevir
    y_pred_proba = y_pred_proba_tensor.numpy()
    
    # Olasılık 0.5'ten büyükse 1 (Yaşadı), küçükse 0 (Öldü) de
    y_pred_class = (y_pred_proba > 0.5).astype(int)

# Skorları Hesapla
py_acc = accuracy_score(y_test, y_pred_class)
py_f1 = f1_score(y_test, y_pred_class)
py_auc = roc_auc_score(y_test, y_pred_proba)

# Sonuçları Kaydet
results['PyTorch Custom NN'] = {'Accuracy': py_acc, 'F1': py_f1, 'AUC': py_auc, 'CV_Accuracy': 'N/A'}

print(f"\n🏆 PyTorch Model Sonuçları:")
print(f" Accuracy: {py_acc:.4f}")
print(f" F1 Score: {py_f1:.4f}")
print(f" AUC Score: {py_auc:.4f}")


In [ ]:
#logistic regression - ML
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_acc = accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_proba)
lr_cv = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='accuracy').mean()
results['Logistic Regression'] = {'Accuracy': lr_acc, 'F1': lr_f1, 'AUC': lr_auc, 'CV_Accuracy': lr_cv}
print(f" Accuracy: {lr_acc:.4f} | F1: {lr_f1:.4f} | AUC: {lr_auc:.4f} | CV: {lr_cv:.4f}")

In [ ]:
# --- Model 2: Random Forest ---
from sklearn.ensemble import RandomForestClassifier
print("\n Model 2: Random Forest")
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_split=5,
random_state=42)
rf_model.fit(X_train, y_train) # RF ölçekleme gerektirmez
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)
rf_cv = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy').mean()
results['Random Forest'] = {'Accuracy': rf_acc, 'F1': rf_f1, 'AUC': rf_auc, 'CV_Accuracy': rf_cv}
print(f" Accuracy: {rf_acc:.4f} | F1: {rf_f1:.4f} | AUC: {rf_auc:.4f} | CV: {rf_cv:.4f}")

In [ ]:
# --- Model 3: XGBoost ---
import xgboost as xgb
from xgboost import XGBClassifier
print("\n Model 3: XGBoost")
xgb_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
use_label_encoder=False, eval_metric='logloss',
random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_proba)
xgb_cv = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy').mean()
results['XGBoost'] = {'Accuracy': xgb_acc, 'F1': xgb_f1, 'AUC': xgb_auc, 'CV_Accuracy': xgb_cv}
print(f" Accuracy: {xgb_acc:.4f} | F1: {xgb_f1:.4f} | AUC: {xgb_auc:.4f} | CV: {xgb_cv:.4f}")

In [ ]:
print("\n" + "="*60)
print(" MODEL DEĞERLENDİRME")
print("="*60)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report

# --- 1. Sonuç Karşılaştırma Tablosu ---
results_df = pd.DataFrame(results).T
# N/A değerleri olduğu için round fonksiyonu hata vermesin diye önce numeric yapmaya çalışıyoruz
# Sadece sayısal sütunları yuvarla
results_df = results_df.apply(pd.to_numeric, errors='ignore').round(4)

print("\n MODEL KARŞILAŞTIRMA TABLOSU:")
print(results_df.to_string())

# --- 2. Görselleştirmeler ---
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle(' Model Değerlendirme Sonuçları', fontsize=16, fontweight='bold')

# --- A) Model Karşılaştırma Bar Chart (Hata Düzeltildi) ---
models = list(results.keys())
metrics = ['Accuracy', 'F1', 'AUC', 'CV_Accuracy']
x = np.arange(len(models))
width = 0.2
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for i, metric in enumerate(metrics):
    values = []
    for m in models:
        val = results[m].get(metric, 0)
        # KRİTİK DÜZELTME: Eğer değer 'N/A' ise grafik çizilebilmesi için 0 yap
        if isinstance(val, str): 
            val = 0.0
        values.append(val)
        
    axes[0, 0].bar(x + i*width, values, width, label=metric, color=colors[i], edgecolor='black')

axes[0, 0].set_xlabel('Modeller')
axes[0, 0].set_ylabel('Skor')
axes[0, 0].set_title('Model Performans Karşılaştırması')
axes[0, 0].set_xticks(x + width * 1.5)
axes[0, 0].set_xticklabels(models, rotation=15)
axes[0, 0].legend()
axes[0, 0].set_ylim(0.5, 1.05) # Skalayı biraz genişlettik
axes[0, 0].grid(axis='y', linestyle='--', alpha=0.7)

# --- B) ROC Curves (PyTorch Dahil) ---
# Mevcut olasılıkları bir sözlükte toplayalım
all_proba = {}
if 'Logistic Regression' in results: all_proba['Logistic Regression'] = lr_proba
if 'Random Forest' in results: all_proba['Random Forest'] = rf_proba
if 'XGBoost' in results: all_proba['XGBoost'] = xgb_proba
if 'Neural Network' in results: all_proba['Neural Network'] = nn_proba

# PyTorch olasılıkları hafızadaysa ekle
try:
    if 'y_pred_proba' in locals():
        all_proba['PyTorch Custom NN'] = y_pred_proba
except:
    pass

for name, proba in all_proba.items():
    if name in models: # Sadece results içinde olanları çiz
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        axes[0, 1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.500)')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Eğrileri')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# --- C) En İyi Modelin Confusion Matrix'i ---
# En yüksek Accuracy'ye sahip modeli bul (Sayısal karşılaştırma için N/A'ları geçici olarak 0 say)
best_model_name = results_df['Accuracy'].astype(float).idxmax()

# Tahminleri topla
all_preds = {}
if 'Logistic Regression' in results: all_preds['Logistic Regression'] = lr_pred
if 'Random Forest' in results: all_preds['Random Forest'] = rf_pred
if 'XGBoost' in results: all_preds['XGBoost'] = xgb_pred
if 'Neural Network' in results: all_preds['Neural Network'] = nn_pred
# PyTorch tahminleri (Class)
try:
    if 'y_pred_class' in locals():
        all_preds['PyTorch Custom NN'] = y_pred_class
except:
    pass

if best_model_name in all_preds:
    best_pred = all_preds[best_model_name]
    cm = confusion_matrix(y_test, best_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
                xticklabels=['Öldü', 'Yaşadı'], yticklabels=['Öldü', 'Yaşadı'])
    axes[1, 0].set_xlabel('Tahmin')
    axes[1, 0].set_ylabel('Gerçek')
    axes[1, 0].set_title(f'Confusion Matrix - Şampiyon: {best_model_name}')

# --- D) Feature Importance (Sadece Random Forest varsa) ---
if 'Random Forest' in results:
    # X.columns yerine X_train.columns kullanmak daha güvenlidir
    feature_imp = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    feature_imp.tail(15).plot(kind='barh', ax=axes[1, 1], color='#3498db', edgecolor='black')
    axes[1, 1].set_title('Feature Importance (Random Forest - Top 15)')
    axes[1, 1].set_xlabel('Önem Skoru')
else:
    axes[1, 1].text(0.5, 0.5, 'Random Forest Modeli Bulunamadı', 
                    horizontalalignment='center', verticalalignment='center', fontsize=12)


# --- Classification Report ---
if best_model_name in all_preds:
    print(f"\n🏆 ŞAMPİYON MODEL: {best_model_name}")
    print(f"Detaylı Performans Raporu:")
    print(classification_report(y_test, all_preds[best_model_name], target_names=['Öldü', 'Yaşadı']))

In [ ]:
print("\n" + "="*60)
print(" HİPERPARAMETRE OPTİMİZASYONU (GridSearchCV)")
print("="*60)

In [ ]:
import warnings
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

# 1. SUSTURUCU: Gereksiz uyarıları engelle
warnings.filterwarnings('ignore') 

# 2. Parametre Havuzu
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_child_weight': [1, 3, 5]
}

print("\n XGBoost hiperparametre araması başlıyor...")

# 3. Grid Search Kurulumu
# DÜZELTME: 'use_label_encoder=False' parametresini sildik.
grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42), 
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1 # Sadece süreç bilgisini versin
)

# 4. Eğitimi Başlat
# Scaled veri veya normal veri fark etmez, hangisi elindeyse
try:
    grid_search.fit(X_train_scaled, y_train)
except NameError:
    grid_search.fit(X_train, y_train)

# 5. Sonuçlar
print(f"\n✅ En iyi parametreler: {grid_search.best_params_}")
print(f"✅ En iyi CV skoru: {grid_search.best_score_:.4f}")

# En iyi modeli kaydet
best_xgb_model = grid_search.best_estimator_

In [ ]:
# En iyi model ile test
best_xgb = grid_search.best_estimator_
best_pred = best_xgb.predict(X_test_scaled)
best_acc = accuracy_score(y_test, best_pred)
print(f" Test accuracy (optimized): {best_acc:.4f}")

In [ ]:
# 1. Senin yazdığın Feature Engineering kısımları (AYNEN KALSIN, DOĞRU)
test['Title'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Title'].map(title_mapping).fillna('Rare')
test['Family_Size'] = test['SibSp'] + test['Parch'] + 1
test['Is_Alone'] = (test['Family_Size'] == 1).astype(int)
test['Has_Cabin'] = test['Cabin'].notna().astype(int)

# 2. Eksik Değerleri Doldurma (AYNEN KALSIN, MANTIK DOĞRU)
# (Train verisinden öğrendiklerinle dolduruyorsun, süper)
for title in test['Title'].unique():
    # Train setinden medyanı alıyoruz (Hata almamak için kontrol ekledik)
    if len(train.loc[train['Title'] == title, 'Age']) > 0:
        median_age = train.loc[train['Title'] == title, 'Age'].median()
        test.loc[(test['Age'].isnull()) & (test['Title'] == title), 'Age'] = median_age

test['Age'] = test['Age'].fillna(train['Age'].median())
test['Fare'] = test['Fare'].fillna(train['Fare'].median())
test['Embarked'] = test['Embarked'].fillna(train['Embarked'].mode()[0])

# 3. YENİ EKLENEN KISIM: Encoding (Get Dummies)
# Kategorik verileri sayıya çevirmeliyiz
test = pd.get_dummies(test, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

# 4. YENİ EKLENEN KISIM: Sütun Hizalama (Alignment)
# Burası Hayati Önem Taşıyor!
# Test setinde olmayan sütunları 0 ile doldur, fazlalıkları at.
# X.columns senin eğitimde kullandığın final sütun listesidir.
test_features = test.reindex(columns=X.columns, fill_value=0)

# 5. Scaling (DÖNÜŞTÜRME)
# Artık sütunlar birebir aynı olduğu için hata vermez
test_scaled = scaler.transform(test_features)

# 6. Tahmin ve Kayıt
predictions = best_xgb.predict(test_scaled)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)
print("✅ Dosya başarıyla oluşturuldu!")
print(submission.tail())
print(f"\nToplam tahmin: {len(submission)}")